# 07_BERT_Model.ipynb
## FakeJobShield: BERT Fine-Tuning
This notebook implements Fine-Tuning of the HuggingFace `bert-base-uncased` transformer on the job text data. To ensure fast execution on CPU, we train on a representative subset of data while providing full training pipeline implementation.


In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import os

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


C:\Users\Rudra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load cleaned text data
df = pd.read_csv("data/cleaned_fake_job_postings.csv")
df["cleaned_text"] = df["cleaned_text"].fillna("")
df["fraudulent_int"] = df["fraudulent"].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0}).fillna(0).astype(int)

# Use subset of data for fast CPU execution
genuine_sub = df[df["fraudulent_int"] == 0].sample(n=100, random_state=42)
fraudulent_sub = df[df["fraudulent_int"] == 1].sample(n=50, random_state=42)
subset_df = pd.concat([genuine_sub, fraudulent_sub]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Subset shape:", subset_df.shape)


Subset shape: (150, 21)


In [3]:
# Split subset into train/test
train_df = subset_df.sample(frac=0.8, random_state=42)
test_df = subset_df.drop(train_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

print(f"Train size: {train_df.shape[0]} | Test size: {test_df.shape[0]}")


Train size: 120 | Test size: 30


In [4]:
# Tokenize dataset using BERT Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode_texts(texts):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

print("Encoding train/test texts...")
train_encodings = encode_texts(train_df["cleaned_text"])
test_encodings = encode_texts(test_df["cleaned_text"])
print("Encoding complete!")


Encoding train/test texts...
Encoding complete!


In [5]:
# Custom PyTorch Dataset
class JobDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
        
    def __len__(self):
        return len(self.labels)

train_dataset = JobDataset(train_encodings, train_df["fraudulent_int"].values)
test_dataset = JobDataset(test_encodings, test_df["fraudulent_int"].values)


In [6]:
# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [7]:
# Load pre-trained BERT
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6289.12it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Define HuggingFace Trainer arguments
training_args = TrainingArguments(
    output_dir="./results/bert_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    use_cpu=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
# Fine-tune model
print("Training BERT...")
trainer.train()
print("BERT Training complete!")


Training BERT...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.646358,0.525727,0.866667,1.000000,0.428571,0.600000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.12it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]

BERT Training complete!


In [10]:
# Evaluate model
eval_results = trainer.evaluate()
print("BERT Evaluation Results:")
print(eval_results)

# Save metrics
import json
with open("results/bert_metrics.json", "w") as f:
    json.dump(eval_results, f, indent=2)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.646358,0.525727,1,0.866667,1.000000,0.428571,0.600000


BERT Evaluation Results:
{'eval_loss': 0.525726854801178, 'eval_accuracy': 0.8666666666666667, 'eval_precision': 1.0, 'eval_recall': 0.42857142857142855, 'eval_f1': 0.6}


In [11]:
# Save BERT model
os.makedirs("models/bert_model", exist_ok=True)
model.save_pretrained("models/bert_model")
tokenizer.save_pretrained("models/bert_model")
print("Saved BERT model weights and tokenizer to 'models/bert_model/'")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.15it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.12it/s]

Saved BERT model weights and tokenizer to 'models/bert_model/'
